# Flash-Next on Colab — ph0

Qwen3.8 Flash-Next を Colab の **A100 80GB / High RAM** で起動し、Pi 用 OpenAI Chat Completions API の応答・ツール往復を検証します。上から順に実行してください。

- GPU 40GB、RAM不足、空きディスク不足の場合は最初のセルで停止します。GPU割当は Colab の **ランタイム → ランタイムのタイプを変更** で設定してください。
- 約100GiBのモデルを取得します。Colab のコンピューティングユニットと時間を消費します。
- ノートブックの実行結果に API キーが残るので、キーを表示した後のノートブックを公開・再保存しないでください。リポジトリ版の出力欄は空です。
- このノートブックはColab内でサーバーを起動します。`google-colab-cli` は不要です。終了時は **ランタイム → 接続解除してランタイムを削除** を実行します。


In [ ]:
import shutil, subprocess
from pathlib import Path

content = Path('/content')
assert content.exists(), 'Google Colab 上で実行してください'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True).strip().splitlines()[0]
name, vram_mib = gpu.rsplit(',', 1)
ram_kib = next(int(line.split()[1]) for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:'))
free_gib = shutil.disk_usage('/content').free / 2**30
print(f'GPU: {name.strip()} / {int(vram_mib):,} MiB | RAM: {ram_kib/2**20:.1f} GiB | disk free: {free_gib:.1f} GiB')
assert int(vram_mib) >= 75*1024, 'A100 80GB が必要です。40GB環境では実行できません'
assert ram_kib / 2**20 >= 120, 'High RAM のランタイムが必要です'
assert free_gib >= 110, '空きディスクが110GiB以上必要です'


## 1. 固定版のコードを取得

上流の `architectds/collabosm` を固定コミットで取得し、このリポジトリのPi向けパッチをSHA-256検証後に適用します。パッチにはChat Completionsのtool calling、モデル用テンプレートへの履歴変換、Jinja依存関係が含まれます。

In [ ]:
import hashlib, urllib.request

UPSTREAM = '138ea8cd9b8edd030f26d945deae19fad8cf7c6d'
PATCH_SHA256 = '07119dacd7852303714521a99bd653da779626d19be575346aa45c75982c7c96'
PATCH_URL = 'https://raw.githubusercontent.com/workoushr/workou-colab/main/patches/pi-tool-calling.patch'
repo = Path('/content/collabosm')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/architectds/collabosm.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '--quiet', 'origin', UPSTREAM], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--force', '--detach', UPSTREAM], check=True)
# A previous run leaves this new (untracked upstream) file behind.
(repo / 'scripts' / 'tool_protocol.py').unlink(missing_ok=True)
patch = urllib.request.urlopen(PATCH_URL, timeout=30).read()
assert hashlib.sha256(patch).hexdigest() == PATCH_SHA256, 'パッチのSHA-256が一致しません'
patch_path = Path('/content/pi-tool-calling.patch')
patch_path.write_bytes(patch)
subprocess.run(['git', '-C', str(repo), 'apply', '--check', str(patch_path)], check=True)
subprocess.run(['git', '-C', str(repo), 'apply', str(patch_path)], check=True)
for filename in ('api_server.py', 'tool_protocol.py', 'bootstrap.sh', 'serve.sh'):
    shutil.copy2(repo / 'scripts' / filename, Path('/content') / filename)
print('Pinned upstream + Pi tool-calling patch ready:', UPSTREAM[:12])


## 2. ランタイムとモデルを準備

最初の実行ではExLlamaV3を導入し、固定リビジョンの重みをHugging Faceから取得します。数分以上かかる場合があります。同じランタイムでの再実行では既存ファイルを利用します。ph0ではKV 262,144トークン、RAM側キャッシュ各8GBから始めます。

In [ ]:
import os, sys, time, torch

wheel_abi = (sys.version_info[:2] == (3, 13) and torch.__version__.split('+')[0] == '2.11.0' and torch.version.cuda == '12.8')
print('Runtime:', 'prebuilt wheel' if wheel_abi else 'source build (runtime ABI differs from pinned wheel)')
os.environ.update({
    'RUNTIME': 'wheel' if wheel_abi else 'source',
    'MODEL_REPO': 'turboderp/Qwen3.8-Flash-Next-exl3',
    'MODEL_REVISION': '55a732e0c4c3d4614bc42b68493bb930d9b02c0a',
    'MODEL_DIR': '/content/exl3',
    'CACHE_SIZE': '262144',
    'CACHE_QUANT': '4',
    'CPU_CACHE_GB': '8',
    'RECURRENT_CACHE_GB': '8',
    'GCS': '4096',
    'NDT': '4',
    'PORT': '8090',
})
t0 = time.monotonic()
subprocess.run(['bash', '/content/bootstrap.sh'], check=True, env=os.environ.copy())
print(f'Bootstrap completed in {time.monotonic()-t0:.0f}s')


## 3. APIと一時トンネルを起動

モデルロードに数分かかります。トンネルURLは起動ごとに変わります。公開URLにはBearer認証が必要です。

In [ ]:
import re

t0 = time.monotonic()
run = subprocess.run(['bash', '/content/serve.sh'], capture_output=True, text=True, timeout=900, env=os.environ.copy())
key_file = Path('/content/api-key.txt')
key = key_file.read_text().strip() if key_file.exists() else ''
print((run.stdout + run.stderr).replace(key, '[API KEY HIDDEN]') if key else run.stdout + run.stderr)
if run.returncode:
    print('serve.log tail:', '\n'.join(Path('/content/serve.log').read_text().splitlines()[-30:]))
    raise RuntimeError(f'serve.sh exited {run.returncode}')
assert key, 'API key was not created'
base_url = Path('/content/url.txt').read_text().strip()
assert base_url.startswith('https://'), 'Tunnel URL was not created; see /content/tunnel.log'
print(f'Endpoint: {base_url}/v1 | ready in {time.monotonic()-t0:.0f}s')


## 4. Chatと外部到達性を検証

認証付きの公開URLでモデル一覧と短いChat Completionsを試します。JSON本文やキーは保存しません。

In [ ]:
import json
from urllib.request import Request, urlopen

MODEL = 'qwen3.8-flash-next-exl3'
def api(path, payload=None, timeout=180):
    headers={'Authorization': f'Bearer {key}', 'Content-Type': 'application/json'}
    data = json.dumps(payload).encode() if payload is not None else None
    request = Request(base_url + '/v1' + path, data=data, headers=headers)
    with urlopen(request, timeout=timeout) as response:
        return json.load(response)

models = api('/models')
assert any(m['id'] == MODEL for m in models['data'])
t0 = time.monotonic()
chat = api('/chat/completions', {
    'model': MODEL, 'messages': [{'role': 'user', 'content': '一文で自己紹介してください。'}],
    'max_tokens': 256, 'stream': False,
})
answer = chat['choices'][0]['message']['content']
assert answer.strip(), 'Chat returned empty text'
print(f'External API: PASS | Chat: {time.monotonic()-t0:.1f}s | {answer[:160]}')


## 5. ツール呼び出しを検証

モデルに `read` を選ばせ、架空のファイル内容を返して次ターンまで試します。**実ファイルをColabで読み書きする試験ではありません。** Piが後でローカル環境でツールを実行するための通信経路を確認します。モデルがツールを選ばなかった場合はph0の検証結果として失敗を表示します。

In [ ]:
tool = {'type':'function', 'function':{
    'name':'read', 'description':'Read a file on the client machine',
    'parameters': {'type':'object', 'properties': {'path': {'type':'string'}},
                   'required':['path'], 'additionalProperties':False}}}
messages = [
    {'role':'system','content':'You are a coding agent. Use the available read tool when asked to read a file.'},
    {'role':'user','content':'Read sample.txt with the read tool, then summarize its contents.'},
]
t0 = time.monotonic()
first = api('/chat/completions', {'model':MODEL, 'messages':messages, 'tools':[tool], 'max_tokens':2048})
msg = first['choices'][0]['message']
calls = msg.get('tool_calls') or []
assert first['choices'][0]['finish_reason'] == 'tool_calls' and calls, f'Model did not call read: {msg.get("content", "")[:200]}'
assert calls[0]['function']['name'] == 'read'
args = json.loads(calls[0]['function']['arguments'])
assert args.get('path') == 'sample.txt', args
messages += [msg, {'role':'tool','tool_call_id':calls[0]['id'],'content':'Sample content: ph0 test passed.'}]
second = api('/chat/completions', {'model':MODEL, 'messages':messages, 'tools':[tool], 'max_tokens':2048})
followup = second['choices'][0]['message'].get('content') or ''
assert followup.strip(), 'Model did not answer after the tool result'
print(f'Tool call + result replay: PASS | {time.monotonic()-t0:.1f}s | {followup[:200]}')


## 6. Pi側の接続設定

Piを動かすPCの `~/.pi/agent/models.json` にREADMEの設定を追加し、そのPCで `COLLABOSM_API_KEY` を設定します。以下はこのランタイムでの接続値です。キー表示後の出力を含むノートブックをGitHubに保存しないでください。

In [ ]:
print('baseUrl =', base_url + '/v1')
print('model =', MODEL)
print('COLLABOSM_API_KEY =', key)


## 終了

Colabの **ランタイム → 接続解除してランタイムを削除** を選びます。セルの実行を止めるだけではランタイムの課金が続く場合があります。ランタイム削除後はトンネルURLと重みが失われます。